# End to end 1 — Identify: naive, adjusted, instrumented, and priced

**The question.** A treatment `X` was applied at varying doses across units and an outcome
`Y` was recorded. Someone has regressed `Y` on `X` and reports a slope. Is that slope the
causal effect of `X` on `Y`, and if not, what *would* be?

This notebook walks the decision an analyst has to make before any estimator runs: declare
the causal graph, ask `axiom.identify` which route (if any) the graph licenses, estimate along
that route, and — when the graph says the effect is *not* identified — price how strong a
hidden confounder would have to be to explain the naive number away.

Three worlds from `axiom.sim` carry their own truth (the effect of `X` on `Y` is `2.0` in all
of them), so every estimate below can be judged against a known answer:

| world | graph | what a naive regression gets wrong |
|---|---|---|
| `confounded_world` | `Z -> X, Z -> Y, X -> Y`, `Z` measured | biased by the open back-door path, fixable by adjusting for `Z` |
| `hidden_confounder_world` | same graph, `Z` unmeasured | biased by the same amount, and **nothing measured fixes it** |
| `iv_world` | `Z -> X, X -> Y, X <-> Y` | biased by the latent common cause; `Z` is an instrument |

Crossed subpackages: `sim` → `identify` → `diagnose`.

In [ ]:
import numpy as np

from axiom.core import is_failure
from axiom.diagnose import BiasBounds, RobustnessValue, bias_bounds, robustness_value, tipping_point
from axiom.identify import (
    CausalGraph, IdentificationVerdict, LinearEstimate, assign_roles, identify, minimal_adjustment_sets, ols,
    two_stage_least_squares, weak_instrument_check,
)
from axiom.sim import confounded_world, hidden_confounder_world, iv_world

from axiom.display import enable, table
from axiom.viz import causal_graph

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, caption, curve_band, intervals, mark_x, mark_y

enable();  # every axiom result renders itself from here on

N, SEED = 5_000, 0

## 1. Declare the graph, read the roles

The graph is the analyst's statement of what causes what. Before asking for an effect, look
at what each node *is* relative to `X -> Y`: `assign_roles` classifies every node (confounder,
mediator, collider, instrument, ...) and the result carries the graph's content hash, so the
classification can be traced back to exactly the graph that produced it.

In [ ]:
world = confounded_world()
print(world.graph.to_text(), "| measured:", sorted(world.graph.measured))
roles = assign_roles(world.graph, "X", "Y")
print("roles:", roles.roles)
print("minimal adjustment sets:", minimal_adjustment_sets(world.graph, "X", "Y"))
truth = world.total_effect("X", "Y")
print("truth (effect of X on Y):", truth)
causal_graph(world.graph, height=300)

## 2. The verdict

`identify` tries back-door, front-door, instrument, and back-door-with-an-unmeasured-node
routes in preference order. The verdict is a `Spec`: its `status`, the `route` it chose, the
`adjustment_set`, and the `assumptions` that route rests on, each with a `state` that says
whether the graph establishes it or whether it is something the data (or the analyst) still
has to supply.

In [ ]:
v: IdentificationVerdict = identify(world.graph, "X", "Y")
print("status:", v.status, "| route:", v.route, "| adjust for:", v.adjustment_set)
table(
    [[a.name, a.state] for a in v.verdict.assumptions],
    headers=("assumption", "state"),
)

## 3. Naive versus back-door-adjusted

The world simulates observational data; `observed()` drops unmeasured columns (none here).
The naive OLS slope carries the confounding bias the docstring predicts,
`2.0 + 1.5 · 0.8 / 1.64 ≈ 2.73`; adjusting for the verdict's set recovers the truth within
its standard error. Every estimate is a `LinearEstimate` whose `ci` is labelled `wald` —
a frequentist interval is a different object from a credible one, and the type says so.

In [ ]:
frame = world.observed(world.simulate(N, seed=SEED))
naive: LinearEstimate = ols(frame, "Y", "X")
adjusted: LinearEstimate = ols(frame, "Y", "X", covariates=v.adjustment_set)


def show(label: str, e: LinearEstimate) -> None:
    ci = e.ci(0.95)
    print(f"{label:22s} {e.estimate:6.3f} ± {e.se:.3f}  [{ci.lower:6.3f}, {ci.upper:6.3f}]  "
          f"err={e.estimate - truth:+.3f}  method={e.method} covariates={e.covariates}")


show("naive", naive)
show("back-door adjusted", adjusted)
print("predicted naive bias:", round(1.5 * 0.8 / 1.64, 3))

## 4. The same world with the confounder hidden

Now `Z` exists but was never recorded. The graph is the same; the *measured* graph is not.
`identify` does not pretend: the verdict is `downgraded`, the route is a back-door set that
needs the unmeasured node, and the verdict names which node. Nothing downstream will report
a point estimate for this effect without an explicit, ledgered `assume_identified`.

In [ ]:
hidden = hidden_confounder_world()
vh = identify(hidden.graph, "X", "Y")
print("status:", vh.status, "| route:", vh.route, "| needs unmeasured:", vh.unmeasured_required)
print("assumptions:", [f"{a.name}:{a.state}" for a in vh.verdict.assumptions])
print("measured columns:", hidden.observed(hidden.simulate(3, seed=0)).columns.tolist())

The only regression the observed columns allow is the naive one, and it is biased by the
same `~0.73` — the number looks as precise as before, which is exactly the trap.

In [ ]:
frame_h = hidden.observed(hidden.simulate(N, seed=SEED))
naive_h = ols(frame_h, "Y", "X")
show("naive (Z hidden)", naive_h)

## 5. A blocked verdict

When the graph has a latent common cause of `X` and `Y` and *no* instrument, mediator, or
measured back-door set, there is no route at all. The verdict is `blocked` and its `reason`
says why. This is the honest answer; the parent project's failure mode was to return a
regression coefficient here anyway.

In [ ]:
truly_blocked = CausalGraph.from_edges("X -> Y, X <-> Y")
vb = identify(truly_blocked, "X", "Y")
print("status:", vb.status, "| route:", vb.route, "|", vb.verdict.reason)

## 6. An instrument

`iv_world` has a latent `X <-> Y` path but also `Z -> X` with no other path from `Z` to
`Y`. The graph licenses the instrument route — `downgraded`, not `identified`, because
exclusion is graphical but **relevance** is a data question and effect homogeneity is not
something a graph can establish. `two_stage_least_squares` recovers the truth;
`weak_instrument_check` turns the first-stage F into the relevance assumption's state.

In [ ]:
ivw = iv_world()
vi = identify(ivw.graph, "X", "Y")
print("status:", vi.status, "| route:", vi.route, "| instrument:", vi.instrument)
print("assumptions:", [f"{a.name}:{a.state}" for a in vi.verdict.assumptions])

frame_iv = ivw.observed(ivw.simulate(N, seed=SEED + 1))
truth_iv = ivw.total_effect("X", "Y")
naive_iv = ols(frame_iv, "Y", "X")
iv = two_stage_least_squares(frame_iv, "Y", "X", instruments=[vi.instrument])
rows = []
for label, e in (("naive", naive_iv), ("2SLS", iv)):
    ci = e.ci(0.95)
    rows.append([label, f"{e.estimate:.3f}", f"{e.se:.3f}",
                 f"[{ci.lower:.3f}, {ci.upper:.3f}]", f"{e.estimate - truth_iv:+.3f}"])
table(rows, headers=("estimator", "estimate", "se", "95%", "error against truth"))
relevance = weak_instrument_check(iv)
print(f"first-stage F = {iv.detail['first_stage_f']:.1f} -> {relevance.name}: {relevance.state}")

In [ ]:
rows = [
    ("Z measured · naive", naive, truth),
    ("Z measured · adjusted for Z", adjusted, truth),
    ("Z hidden · naive (all you can run)", naive_h, truth),
    ("latent path · naive", naive_iv, truth_iv),
    ("latent path · 2SLS on the instrument", iv, truth_iv),
]
fig = intervals(
    [(label, e.estimate - tr, e.ci(0.95).lower - tr, e.ci(0.95).upper - tr) for label, e, tr in rows],
    ref=0.0, ref_label="the truth",
    highlight="Z hidden · naive (all you can run)",
    title="Five regressions, one true effect of 2.0",
    subtitle="estimate minus truth, with 95% Wald intervals — same data, different verdicts",
    x_title="error",
)
caption(fig, "The two rows that sit on the line are the ones the graph licensed. The "
             "highlighted row is the one with no licensed alternative: same bias as the row "
             "above it, same precision, and nothing in the data to fix it. That is the case "
             "the rest of this notebook is about.")

## 7. Pricing the hidden confounder

Back to the hidden-confounder world, where the naive slope is all we have. Rather than
report it, ask the sensitivity question (Cinelli & Hazlett 2020): how strong would an
unobserved confounder have to be — as a partial R² with both treatment and outcome — to
reduce the estimate to zero (`RV_1`), or to make it no longer significant (`RV_{1,α}`)?
`bias_bounds` then prices a *stated* confounder strength into an adjusted estimate and
interval.

We know the truth here, so we can also check: the confounder that actually exists would
need to remove `0.73` of a `2.73` estimate, i.e. `q ≈ 0.27`.

In [ ]:
df = naive_h.n - 2  # intercept and slope
rv = robustness_value(estimate=naive_h.estimate, se=naive_h.se, df=df, q=1.0, alpha=0.05)
assert isinstance(rv, RobustnessValue)
print(f"RV_1 = {rv.rv:.3f}  RV_1,0.05 = {rv.rv_alpha:.3f}  (partial R2 of X with Y: {rv.r2_yd_x:.3f})")

q_true = (naive_h.estimate - truth) / naive_h.estimate
rv_q = robustness_value(estimate=naive_h.estimate, se=naive_h.se, df=df, q=q_true, alpha=0.05)
assert isinstance(rv_q, RobustnessValue)
print(f"to remove the actual bias (q={q_true:.2f}) a confounder needs RV_q = {rv_q.rv:.3f}")

In this linear world the hidden `Z` has a known strength: its partial R² with `X` is
`0.64 / 1.64 ≈ 0.39` and its partial R² with `Y` given `X` follows from the coefficients.
Plugging a confounder of roughly that strength into `bias_bounds` recovers the truth; plugging
in a weaker one does not. The adjusted interval carries its mass, as every interval must.

In [ ]:
# Partial R2 of Z with X (Z is the only other cause of X): var(0.8 Z) / var(X) = 0.64 / 1.64.
r2_dz_x = 0.64 / 1.64
# Partial R2 of Z with Y given X: from the residual of Y on X, which is 1.5 Z (orthogonalized on X) + noise.
resid_z = 1.5 ** 2 * (1.0 - r2_dz_x)
r2_yz_dx = resid_z / (resid_z + 1.0)
print(f"implied confounder strength: r2_dz_x={r2_dz_x:.3f}  r2_yz_dx={r2_yz_dx:.3f}")

rows = []
for label, rd, ry in (("actual Z", r2_dz_x, r2_yz_dx), ("half as strong", r2_dz_x / 2, r2_yz_dx / 2)):
    b = bias_bounds(estimate=naive_h.estimate, se=naive_h.se, df=df, r2_yz_dx=ry, r2_dz_x=rd, mass=0.95)
    assert isinstance(b, BiasBounds)
    rows.append([label, f"{b.bias:.3f}", f"{b.adjusted_estimate:.3f}", str(b.adjusted_interval)])
table(rows, headers=("confounder as strong as", "bias", "adjusted", "adjusted interval"))
print("truth:", truth)

## 8. The decision-scale question

Suppose the decision is "dose more if the effect exceeds 1.0 per unit". `tipping_point`
walks a bias grid against draws of the estimate and reports the smallest bias at which the
decision flips. The naive estimate survives the actual confounding here (`2.0 > 1.0`) — but
the analyst now knows *by how much*, instead of trusting a number the graph said was not
identified.

In [ ]:
rng = np.random.default_rng(SEED)
draws = rng.normal(naive_h.estimate, naive_h.se, size=4000)
tp = tipping_point(draws, 1.0, np.linspace(0.0, 2.5, 26), certainty=0.9)
print(f"decision at zero bias: {tp.decision_at_zero} (P={tp.probability_at_zero:.3f})")
print(f"flips at bias {tp.bias}; actual confounding bias = {naive_h.estimate - truth:.3f}; interval at the flip: {tp.interval}")

In [ ]:
bias_grid = np.linspace(0.0, 2.5, 60)
probability = [float(np.mean(draws - b > 1.0)) for b in bias_grid]
fig = curve_band(
    bias_grid, probability,
    label="P(effect > 1.0)",
    title="How much hidden confounding the decision can absorb",
    subtitle="probability the effect clears the decision threshold, against the bias assumed",
    x_title="assumed bias", y_title="probability",
)
mark_y(fig, 0.9, text="the certainty the decision needs")
mark_x(fig, float(naive_h.estimate - truth), text="the bias actually present", color=CRITICAL)
mark_x(fig, tp.bias, text="where it flips")
caption(fig, "The confounding that really exists is well to the left of the flip, so this "
             "decision survives it — but that is now a measured margin rather than a hope. "
             "An analyst who reported the naive slope as the effect would have had the same "
             "margin and no way to know it.")

## What this notebook decided

1. With `Z` measured, the graph licenses back-door adjustment and `ols(..., covariates=["Z"])`
   recovers `2.0`; the naive slope is off by the amount the graph predicts.
2. With `Z` hidden, the verdict is *downgraded* and the naive slope is the only estimate —
   so it is reported with its robustness value and a priced bias, not as the effect.
3. With a latent `X <-> Y` and no instrument, the verdict is *blocked* and no number is produced.
4. With an instrument, 2SLS recovers the truth and the first-stage F fills in the one
   assumption the graph could not.

Every number printed above carries the route it came from, the assumptions it needs, and
the interval definition and mass it is stated at.